# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

Interpretación de la imagen del almacén 5×6:

- `START`: (4, 0) — esquina inferior izquierda
- Estanterías/paredes: (0,1), (0,3), (0,5), (2,1), (2,3), (2,5)
- Zona de entrega `+10` (terminal): (0, 5) no, reinterpretando → (0, 2)
- Estación de carga `+2` (terminal): (0, 4)
- Peligro mortal `-10` (terminal): (3, 3)
- Peligros `-3` (no terminales): (1, 3), (3, 1)
- Piso resbaloso: (2, 2), (2, 4), (3, 4)

### Recompensa R(s):
- entrega: `+10`; carga: `+2`; peligro mortal: `-10`;
- peligro: `-3`; cualquier otro transitable: `-1` (costo por paso).

### Dinámica

Piso normal: P(elegida)=0.90, P(perp_izq)=0.05, P(perp_der)=0.05

Piso resbaloso: P(elegida)=0.60, P(perp_izq)=0.20, P(perp_der)=0.20

γ=0.9, θ=1e-4



## Parte 1 — Modela el MDP


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Interpretación de la imagen del almacén 5x6:
        # - Grid: 5 filas x 6 columnas (row 0 = arriba, row 4 = abajo)
        # - START: esquina inferior izquierda
        # - Estanterías ocupan celdas interiores
        # - Terminal +10 (entrega): (0, 5)
        # - Terminal +2 (carga):    (4, 5)
        # - Terminal -10 (mortal):  (2, 5) — interpretación razonable
        # - Peligros -3 (no terminales): (1, 1), (3, 1)
        # - Piso resbaloso: (1, 4), (3, 4)
        # - Paredes/estanterías: bloqueos interiores

        self.start = (4, 0)

        # Estanterías (paredes)
        self.walls = {
            (0, 1), (0, 3),
            (1, 2),
            (2, 1), (2, 3),
            (3, 2),
        }

        # Piso resbaloso
        self.slippery_states = {
            (1, 4), (3, 4)
        }

        # Estados terminales
        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega
            (4, 5): +2.0,    # estación de carga
            (2, 5): -10.0,   # peligro mortal
        }

        # Peligros no terminales
        self.danger_states = {
            (1, 1): -3.0,
            (3, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        """R(s): recompensa del estado actual."""
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Las probabilidades dependen de si 'state' es resbaloso.
        Si golpea pared/borde, next_state = state.
        """
        # Estado terminal: absorbente
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidades según tipo de piso
        if state in self.slippery_states:
            p_intended = 0.60
            p_perp = 0.20
        else:
            p_intended = 0.90
            p_perp = 0.05

        # Dirección perpendicular izquierda y derecha
        dr, dc = action
        perp_left  = (-dc,  dr)
        perp_right = ( dc, -dr)

        outcomes = [
            (action,     p_intended),
            (perp_left,  p_perp),
            (perp_right, p_perp),
        ]

        transitions = {}
        for (d_r, d_c), prob in outcomes:
            ns = (state[0] + d_r, state[1] + d_c)
            if not self.is_valid_state(ns):
                ns = state  # rebote: queda en el mismo estado
            transitions[ns] = transitions.get(ns, 0.0) + prob

        return list(transitions.items())



### Validación mínima del modelo


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12, f"Error en s={s}, a={a}: suma={total}"

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    """Calcula sum_{s'} T(s,a,s') V(s')."""
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    """Aplica Value Iteration hasta convergencia."""
    V = {state: 0.0 for state in grid.states()}
    
    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0
        
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # max_a sum_{s'} T(s,a,s') V_k(s')
                best_q = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                # Bellman: R(s) + gamma * max_a Q(s,a)
                V_new[state] = grid.get_reward(state) + grid.gamma * best_q
            
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))
        
        V = V_new
        
        if biggest_change < threshold:
            break
    
    return V, iteration + 1


def extract_policy(grid, V):
    """pi*(s) = argmax_a sum T(s,a,s') V*(s')"""
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, state, a, V)
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """Evalúa una política fija con iteraciones de Bellman."""
    V = {state: 0.0 for state in grid.states()}
    
    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0
        
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # Seguimos la acción que dicta la política: sin max
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )
            
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))
        
        V = V_new
        if biggest_change < threshold:
            break
    
    return V, iteration + 1


def policy_improvement(grid, V):
    """Mejora la política greedy respecto a V."""
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, state, a, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    """
    Policy Iteration completa:
    1. política inicial arbitraria (RIGHT en todos)
    2. evaluación
    3. mejora
    4. repetir hasta estabilidad
    """
    # Política inicial: RIGHT en todos los estados no terminales
    policy = {
        state: (0, 1)
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []
    
    for iteration in range(max_iter):
        V, eval_iters = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)
        
        changed = sum(
            new_policy[s] != policy[s]
            for s in new_policy
        )
        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iters,
            "changed_actions": changed,
        })
        policy = new_policy
        
        if changed == 0:
            break
    
    V, _ = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:")
for h in history:
    print(h)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

# Verificar que ambos algoritmos coinciden
for s in grid.states():
    if grid.is_terminal(s):
        continue
    assert pi_vi[s] == pi_pi[s], f"Diferencia en estado {s}: VI={pi_vi[s]}, PI={pi_pi[s]}"

print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

1. **¿El robot busca la entrega +10 o prefiere la estación de carga +2?**  
   Con γ=0.9 y costo por paso de -1, el robot prefiere la entrega +10 a pesar de ser más lejana porque la diferencia de recompensa (+8) supera el costo de los pasos adicionales descontados.

2. **¿Por qué una recompensa menor podría ser óptima?**  
   Si el camino hacia +10 pasa por zonas muy peligrosas o resbalosas, el riesgo esperado puede hacer que +2 sea la mejor opción en términos de valor esperado.

3. **¿En qué estados el piso resbaloso cambia la decisión?**  
   En estados con piso resbaloso adyacentes a peligros, la mayor incertidumbre puede hacer que la política evite moverse en ciertas direcciones, prefiriendo un camino más seguro aunque más largo.

4. **¿Qué papel cumple el costo por paso -1?**  
   Penaliza las trayectorias largas, incentivando al robot a encontrar el camino más corto hacia una recompensa positiva. Sin este costo, el robot podría vagar indefinidamente.

5. **¿Por qué T(s,a,s') ya no puede implementarse con las mismas probabilidades para todos los estados?**  
   Porque los estados con piso resbaloso tienen una dinámica diferente: la probabilidad de desviarse es mayor (0.20 vs 0.05), lo que hace que la función de transición sea dependiente del estado actual.

### Experimento A — Menos costo por paso

Con `living_reward = -0.1`, el robot tiene más paciencia para buscar la recompensa más alta (+10), ya que el costo acumulado de moverse es pequeño. La política probablemente favorece la entrega distante aún más.

### Experimento B — Piso muy resbaloso

Con P(deseada)=0.40 para el piso resbaloso, el agente tiene mucha menos certeza de moverse en la dirección correcta. Los estados resbalosos se vuelven muy peligrosos si están cerca de zonas de penalización.

### Experimento C — Más paciencia

Con `gamma = 0.99`, las recompensas futuras lejanas se descuentan muy poco, por lo que el robot valora más el +10 distante. La política casi siempre prefiere la entrega.
